This notebook contains the steps for processing the factor dataframe and merging it with the S&P500 constituents dataframe

In [2]:
import pandas as pd

#path to factor file
file_path = "C:/Users/Admin/projects/Omer_Sen_Thesis_Quantitative_Finance/02_Data_Collection_and_Preperation/Raw_Data/stock-level factors.csv"

try:
    #read the first row of the factor file
    first_row = pd.read_csv(file_path, nrows=1)
    print("Columns Names:")
    print(first_row.columns.tolist())

    print("First row:")
    print(first_row)
except pd.errors.EmptyDataError:
    print("The file is empty")

#read the factor file


Columns Names:
['obs_main', 'exch_main', 'common', 'primary_sec', 'permno', 'date', 'prc', 'market_equity', 'div12m_me', 'chcsho_12m', 'eqnpo_12m', 'ret_1_0', 'ret_3_1', 'ret_6_1', 'ret_9_1', 'ret_12_1', 'ret_12_7', 'ret_60_12', 'seas_1_1an', 'seas_1_1na', 'seas_2_5an', 'seas_2_5na', 'seas_6_10an', 'seas_6_10na', 'seas_11_15an', 'seas_11_15na', 'seas_16_20an', 'seas_16_20na', 'at_gr1', 'sale_gr1', 'capx_gr1', 'inv_gr1', 'debt_gr3', 'sale_gr3', 'capx_gr3', 'inv_gr1a', 'lti_gr1a', 'sti_gr1a', 'coa_gr1a', 'col_gr1a', 'cowc_gr1a', 'ncoa_gr1a', 'ncol_gr1a', 'nncoa_gr1a', 'fnl_gr1a', 'nfna_gr1a', 'tax_gr1a', 'be_gr1a', 'ebit_sale', 'gp_at', 'cop_at', 'ope_be', 'ni_be', 'ebit_bev', 'netis_at', 'eqnetis_at', 'dbnetis_at', 'oaccruals_at', 'oaccruals_ni', 'taccruals_at', 'taccruals_ni', 'noa_at', 'opex_at', 'at_turnover', 'sale_bev', 'rd_sale', 'cash_at', 'sale_emp_gr1', 'emp_gr1', 'ni_inc8q', 'noa_gr1a', 'ppeinv_gr1a', 'lnoa_gr1a', 'capx_gr2', 'saleq_gr1', 'niq_be', 'niq_at', 'niq_be_chg1', 'ni

In [3]:
constituents_file_path = "C:/Users/Admin/projects/Omer_Sen_Thesis_Quantitative_Finance/02_Data_Collection_and_Preperation/Raw_Data/sp500_constituents.csv"

#s&p constituents file
sp500_df = pd.read_csv(constituents_file_path)
print("Successfully loaded S&P500 constituents data")
print(sp500_df.head())

#check if data is end of month
if 'date' in sp500_df.columns:
    sp500_df['date'] = pd.to_datetime(sp500_df['date'])
    # CRSP monthly data is usually end-of-month. Ensure this for consistent merging.
    # If your dates are already EOM, this line won't change them.
    # If they are start/mid-month, this will shift them to EOM.
    sp500_df['date'] = sp500_df['date'] + pd.offsets.MonthEnd(0)
    print("Converted 'date' column in sp500_df to datetime and ensured End-of-Month.")
else:
    print("Warning: 'date' column not found in sp500_constituents.csv. Please check column names.")
        
 # Convert permno to a nullable integer type in sp500_df for robust merging
    if 'permno' in sp500_df.columns:
        try:
            sp500_df['permno'] = pd.to_numeric(sp500_df['permno'], errors='coerce').astype('Int64')
            print("Converted 'permno' column in sp500_df to Int64.")
        except Exception as e:
            print(f"Warning: Could not convert permno to numeric in sp500_df. Error: {e}")
    else:
        print("Warning: 'permno' column not found in sp500_constituents.csv.")

    # Display first few rows to verify
    print("\nFirst 5 rows of sp500_df:")
    print(sp500_df.head())
    print("\nInfo for sp500_df:")
    sp500_df.info()
    

# --- Step 2: Process stock-level factors.csv in Chunks and Merge --- 
# Define chunk size
chunk_size = 1000000  # Adjust as needed based on your memory capacity (e.g., 1,000,000 rows)

# List to hold processed and merged DataFrames
merged_chunks_list = []

if sp500_df is not None: # Proceed only if sp500_df was loaded successfully
    try:
        print(f'\nStarting to process {file_path} in chunks...\n')
        for i, chunk in enumerate(pd.read_csv(file_path, chunksize=chunk_size, na_values=['NaN', '', ' '])):
            print(f"Processing chunk {i+1}...")
            
            # Ensure 'permno' is float or int to handle potential NaNs before converting to int
            if 'permno' in chunk.columns:
                try:
                    chunk['permno'] = pd.to_numeric(chunk['permno'], errors='coerce').astype('Int64')
                except Exception as e:
                    print(f"Warning: Could not convert permno to numeric in factor chunk {i+1}. Error: {e}")
                    continue # Skip chunk if permno conversion fails critically
            else:
                print(f"Warning: 'permno' column not found in chunk {i+1} of factors data.")
                continue 
                
            # Convert date column to datetime objects
            if 'date' in chunk.columns:
                chunk['date'] = pd.to_datetime(chunk['date'], errors='coerce')
                # Set to end of month
                chunk['date'] = chunk['date'] + pd.offsets.MonthEnd(0)
            else:
                print(f"Warning: 'date' column not found in chunk {i+1} of factors data.")
                continue 
            
            # Drop rows where permno or date became NaT/NaN after conversion
            chunk.dropna(subset=['permno', 'date'], inplace=True)
            
            if not chunk.empty:
                # Merge with sp500_df
                merged_chunk = pd.merge(sp500_df, chunk, on=['permno', 'date'], how='inner')
                
                if not merged_chunk.empty:
                    print(f"Chunk {i+1} merged. Shape of merged chunk: {merged_chunk.shape}")
                    merged_chunks_list.append(merged_chunk)
                else:
                    print(f"Chunk {i+1} resulted in an empty merge (no common permno/date combinations with S&P500 data).")
            else:
                print(f"Chunk {i+1} is empty after processing dates/permnos.")
                
          
        if merged_chunks_list:
            # Concatenate all merged chunks
            final_df = pd.concat(merged_chunks_list, ignore_index=True)
            print(f"\nSuccessfully processed and merged data (from tested chunks). Shape of final_df: {final_df.shape}")
            print("\nFirst 5 rows of final_df:")
            final_df.drop(columns=['start', 'ending','obs_main','exch_main', 'common', 'primary_sec', 'age'], inplace=True)
            print(final_df.head())
            print("\nInfo for final_df:")
            final_df.info()
            
            # save the final_df to a new CSV or Parquet file if needed
            final_df.to_parquet('../Proccessed_Data/sp500_factors_merged.parquet', index=False)
            print("\nSaved merged data to sp500_factors_merged.parquet")

        else:
            print("\nNo data was merged. 'merged_chunks_list' is empty. This might be because:")
            print("1. The 'permno' or 'date' values in the first chunk(s) of factors_file.csv did not match any in sp500_constituents.csv.")
            print("2. The date ranges do not overlap after adjustment to End-of-Month.")
            print("3. Issues with data types or missing key values in the initial rows of factors_file.csv.")
            print("Consider checking the date ranges and identifiers in both files if this is unexpected.")

    except FileNotFoundError:
        print(f"Error: The file {file_path} was not found.")
    except Exception as e:
        print(f"An error occurred during chunk processing or merging: {e}")
else:
    print("\nSkipping factor data processing because sp500_constituents.csv could not be loaded.")


Successfully loaded S&P500 constituents data
   permno       start      ending        date       ret
0   20407  1957-03-01  1983-08-31  1965-01-29  0.055901
1   17478  1957-03-01  2024-12-31  1965-01-29  0.105960
2   23528  1957-03-01  1969-03-26  1965-01-29  0.082902
3   17806  1957-03-01  2007-03-19  1965-01-29  0.023091
4   15499  1957-03-01  1969-05-28  1965-01-29 -0.005450
Converted 'date' column in sp500_df to datetime and ensured End-of-Month.

Starting to process C:/Users/Admin/projects/Omer_Sen_Thesis_Quantitative_Finance/02_Data_Collection_and_Preperation/Raw_Data/stock-level factors.csv in chunks...

Processing chunk 1...
Chunk 1 merged. Shape of merged chunk: (1097, 162)
Processing chunk 2...
Chunk 2 merged. Shape of merged chunk: (531, 162)
Processing chunk 3...
Chunk 3 merged. Shape of merged chunk: (1346, 162)
Processing chunk 4...
Chunk 4 merged. Shape of merged chunk: (2188, 162)
Processing chunk 5...
Chunk 5 merged. Shape of merged chunk: (882, 162)
Processing chunk 6

In [12]:
riskfree_rate = pd.read_csv('C:/Users/Admin/projects/Omer_Sen_Thesis_Quantitative_Finance/02_Data_Collection_and_Preperation/Raw_Data/riskfreerates.csv')
riskfree_rate.rename(columns={'dateff': 'date'}, inplace=True)
riskfree_rate['date'] = pd.to_datetime(riskfree_rate['date'])
riskfree_rate['date'] = riskfree_rate['date'] + pd.offsets.MonthEnd(0)

print(riskfree_rate.head())
print(riskfree_rate.tail())
print(riskfree_rate.columns)
print(riskfree_rate.info())







       rf       date
0  0.0028 1965-01-31
1  0.0030 1965-02-28
2  0.0036 1965-03-31
3  0.0031 1965-04-30
4  0.0031 1965-05-31
         rf       date
703  0.0045 2023-08-31
704  0.0043 2023-09-30
705  0.0047 2023-10-31
706  0.0044 2023-11-30
707  0.0043 2023-12-31
Index(['rf', 'date'], dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 708 entries, 0 to 707
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   rf      708 non-null    float64       
 1   date    708 non-null    datetime64[ns]
dtypes: datetime64[ns](1), float64(1)
memory usage: 11.2 KB
None


In [20]:
final_df = pd.merge(final_df, riskfree_rate, on='date', how='left')












In [21]:
final_df['excess_return'] = final_df['ret'] - final_df['rf']












In [23]:
final_df.drop(columns=['ret'], inplace=True)
print(final_df.tail())
print(final_df.columns)










        permno       date          prc  market_equity  div12m_me  chcsho_12m  \
348047   88837 2023-12-31   128.539993   24593.685455   0.022729   -0.001737   
348048   90993 2023-12-31   128.429993   73508.704328   0.012926    0.024728   
348049   93002 2023-12-31  1116.250000  522562.391250   0.015639    0.120260   
348050   88281 2023-12-31   236.399994   12126.610487   0.000000    0.008216   
348051   87055 2023-12-31   660.080017  292895.985823   0.028721   -0.000002   

        eqnpo_12m   ret_1_0   ret_3_1   ret_6_1  ...  bidaskhl_21d  \
348047   0.028500  0.057510  0.161977  0.180250  ...      0.004290   
348048  -0.009626  0.131852  0.034721  0.010564  ...      0.002710   
348049  -0.090912  0.211476  0.114558  0.073121  ...      0.005465   
348050  -0.008182  0.199513  0.005613 -0.062640  ...      0.004564   
348051   0.029896  0.138914  0.050977  0.104912  ...      0.003099   

        corr_1260d  betabab_1260d  rmax5_rvol_21d       qmj  qmj_prof  \
348047    0.725756       

In [11]:
if 'final_df' in locals():
    print("\nNumber of unique constituents per date in final_df:")
    print(final_df.groupby('date')['permno'].nunique().describe())

    # To see a sample of counts per date:
    print("\nSample of counts per date:")
    print(final_df.groupby('date')['permno'].nunique().sample(10))
else:
    print("\nfinal_df is not defined, skipping constituent count per date.")



Number of unique constituents per date in final_df:
count    708.000000
mean     491.598870
std        9.126029
min      469.000000
25%      485.000000
50%      497.000000
75%      498.000000
max      499.000000
Name: permno, dtype: float64

Sample of counts per date:
date
2015-02-28    480
2009-07-31    485
1982-03-31    498
2017-03-31    474
1974-11-30    489
1989-02-28    498
2006-10-31    488
1991-02-28    499
1973-06-30    492
1970-12-31    496
Name: permno, dtype: int64


In [24]:

import numpy as np


if 'final_df' not in locals():
    print("Error: final_df is not loaded. Please load it first.")
    try:
        final_df = pd.read_parquet('02_Data_Collection_and_Preperation/Proccessed_Data/sp500_factors_merged.parquet')
        print("Successfully loaded sp500_factors_merged.parquet")
    except FileNotFoundError:
        print("Could not find sp500_factors_merged.parquet. Please ensure it's in the correct path.")
        final_df = None
    except Exception as e:
        print(f"Error loading Parquet file: {e}")
        final_df = None

if final_df is not None:
    print(f"\nShape of final_df: {final_df.shape}")

    # --- 1. Date Range Confirmation ---
    print("\n--- Date Range Confirmation ---")
    if 'date' in final_df.columns:
        min_date = final_df['date'].min()
        max_date = final_df['date'].max()
        print(f"Date range in final_df: {min_date.strftime('%Y-%m-%d')} to {max_date.strftime('%Y-%m-%d')}")
    else:
        print("'date' column not found.")

    # --- 2. Data Types Check ---
    print("\n--- Data Types Check ---")
    print("Data types of columns:")
    print(final_df.dtypes.value_counts()) # Summary of data types
    print(final_df.info()) 
    
    # Identify non-numeric columns that are not 'date' or 'permno' (or other known identifiers)
    # First, get a list of your actual factor columns.
    # Assuming 'permno', 'date', 'ret' and the first few metadata columns are not factors.
    # The factor columns start after 'ret' based on your previous output (160 columns total)
    # df.columns was: ['permno', 'date', 'ret', 'obs_main', 'exch_main', 'common', 'primary_sec', ...]
    # So, factors might start from the 4th column if sp500_constituents only had permno, date, ret
    # Or, if sp500_constituents had more columns that were kept, adjust the starting point.
    # Based on final_df info: permno, date, ret + 6 int64 columns (likely identifiers) + 153 float64 (factors)
    # Let's assume identifier columns are permno, date, and the int64 ones.
    
    identifier_cols = ['permno', 'date', 'excess_return'] + final_df.select_dtypes(include=['int64']).columns.tolist()
    # Remove duplicates if 'permno' was in int64_cols
    identifier_cols = list(dict.fromkeys(identifier_cols)) 
    
    factor_columns = [col for col in final_df.columns if col not in identifier_cols]
    
    print(f"\nIdentified {len(factor_columns)} potential factor columns.")
    # print("First 5 factor columns:", factor_columns[:5]) # Uncomment to check

    numeric_factor_columns = final_df[factor_columns].select_dtypes(include=np.number).columns.tolist()
    non_numeric_factor_columns = final_df[factor_columns].select_dtypes(exclude=np.number).columns.tolist()

    if not non_numeric_factor_columns:
        print("All identified factor columns are numeric (float64 or int64).")
    else:
        print(f"\nWarning: The following {len(non_numeric_factor_columns)} factor columns are NOT purely numeric:")
        for col in non_numeric_factor_columns:
            print(f"  - {col} (dtype: {final_df[col].dtype})")
        print("These may need further cleaning or type conversion.")

    # --- 3. Missing Values in Factors ---
    print("\n--- Missing Values Check (for factor columns) ---")
    missing_values_summary = final_df[factor_columns].isnull().sum()
    missing_values_summary = missing_values_summary[missing_values_summary > 0].sort_values(ascending=False)
    
    if not missing_values_summary.empty:
        print("Columns with missing values (and their counts):")
        print(missing_values_summary)
        
        total_observations = len(final_df)
        missing_percentage = (missing_values_summary / total_observations) * 100
        print("\nColumns with missing values (and their percentages):")
        print(missing_percentage)
        
        high_missing_cols = missing_percentage[missing_percentage > 20] # Example threshold: 20%
        if not high_missing_cols.empty:
            print(f"\nWarning: The following {len(high_missing_cols)} columns have more than 20% missing values:")
            print(high_missing_cols)
        else:
            print("\nNo factor columns have more than 20% missing values.")
            
    else:
        print("No missing values found in the identified factor columns.")

    # --- 4. Outlier Checks (Descriptive Statistics) ---
    print("\n--- Outlier Check (Descriptive Statistics for numeric factor columns) ---")
    if numeric_factor_columns: # Proceed only if there are numeric factor columns
        desc_stats = final_df[numeric_factor_columns].describe(percentiles=[.01, .05, .25, .5, .75, .95, .99]).transpose()
        print("Descriptive statistics (including 1st, 5th, 95th, 99th percentiles):")
        print(desc_stats[['min', '1%', '5%', '25%', '50%', '75%', '95%', '99%', 'max', 'mean', 'std']])
        
        # Example of identifying potential outliers based on standard deviations (e.g., > 5 std from mean)
        # This is a basic check; more sophisticated methods exist.
        print("\nPotential outlier check (values more than 5 std deviations from the mean - sample):")
        potential_outliers_count = {}
        for col in numeric_factor_columns:
            mean_val = final_df[col].mean()
            std_val = final_df[col].std()
            if std_val > 0: # Avoid division by zero for columns with no variance
                outlier_threshold_upper = mean_val + 5 * std_val
                outlier_threshold_lower = mean_val - 5 * std_val
                num_outliers = final_df[(final_df[col] > outlier_threshold_upper) | (final_df[col] < outlier_threshold_lower)].shape[0]
                if num_outliers > 0:
                    potential_outliers_count[col] = num_outliers
        
        if potential_outliers_count:
            print("Number of potential outliers (more than 5 std from mean) per factor:")
            for col, count in potential_outliers_count.items():
                 print(f"  - {col}: {count} ({((count/total_observations)*100):.2f}%)")
        else:
            print("No factors found with values more than 5 std deviations from the mean using this basic check.")
            
    else:
        print("No numeric factor columns identified to check for outliers.")
        
    print("\n--- Data Processing Checks Complete ---")
    print("Review the missing value counts and descriptive statistics to decide on handling strategies.")
    print("Consider imputation for missing values (e.g., mean, median, model-based) or dropping columns/rows if missingness is too high.")
    print("For outliers, consider winsorization, transformation, or capping if they are influential.")

else:
    print("\nCould not perform checks as final_df was not loaded.")


Shape of final_df: (348052, 156)

--- Date Range Confirmation ---
Date range in final_df: 1965-01-31 to 2023-12-31

--- Data Types Check ---
Data types of columns:
float64           154
int64               1
datetime64[ns]      1
Name: count, dtype: int64
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 348052 entries, 0 to 348051
Columns: 156 entries, permno to excess_return
dtypes: datetime64[ns](1), float64(154), int64(1)
memory usage: 414.2 MB
None

Identified 153 potential factor columns.
All identified factor columns are numeric (float64 or int64).

--- Missing Values Check (for factor columns) ---
Columns with missing values (and their counts):
rd5_at             200426
rd_me              179602
rd_sale            179589
seas_16_20na       107192
seas_16_20an       105819
                    ...  
ret_3_1               311
mispricing_perf       282
ret_1_0               168
market_equity          39
prc                    39
Length: 152, dtype: int64

Columns with missing valu

In [28]:
final_df.drop(columns=['rf'], inplace=True)
print(final_df.head())
print(final_df.tail())
print(final_df.columns)
# integer columns: age, permno
# datetime columns: date


   permno       date     prc  market_equity  div12m_me  chcsho_12m  eqnpo_12m  \
0   23536 1965-01-31  29.250     397.098000   0.035245    0.099984  -0.046160   
1   24328 1965-01-31  43.625     107.186625   0.042390    0.000407   0.045592   
2   22496 1965-01-31  50.500     484.901000   0.022574    0.000000   0.023952   
3   17654 1965-01-31  28.000     319.256000   0.037040    0.012431   0.031046   
4   13688 1965-01-31  35.000    2055.795000   0.031126    0.039998  -0.002262   

    ret_1_0   ret_3_1   ret_6_1  ...  betadown_252d  bidaskhl_21d  corr_1260d  \
0  0.022338  0.026667  0.038481  ...       0.619361      0.005086    0.364707   
1 -0.002857  0.089259  0.078700  ...       0.245008      0.003828    0.341587   
2  0.065963 -0.044110 -0.069164  ...      -0.275304      0.003964    0.434941   
3  0.000000  0.063778  0.166506  ...       0.856719      0.003501    0.459791   
4  0.021898  0.022963  0.114184  ...       0.574610      0.003746    0.535644   

   betabab_1260d  rmax5_rv

In [29]:
##Standardizing factors and handling missing values
from scipy.stats import rankdata


if final_df is not None:
    print(f"\nShape of final_df before imputation and standardization: {final_df.shape}")
    
    # Create a copy to work on
    df_imputed_standardized = final_df.copy()

    # --- Identify factor columns ---
    all_columns = df_imputed_standardized.columns.tolist()
    # Update this list with all known non-factor columns present in your final_df
    known_identifiers_and_target = [
        'permno', 'date', 'excess_return'
    ]
    actual_identifiers = [col for col in known_identifiers_and_target if col in df_imputed_standardized.columns]
    
    # Select factor columns (assuming they are float64 as per previous info)
    factor_columns = [col for col in all_columns if col not in actual_identifiers and df_imputed_standardized[col].dtype == 'float64']
    
    print(f"\nIdentified {len(factor_columns)} factor columns for processing.")
    if not factor_columns:
        print("Warning: No factor columns identified. Check 'known_identifiers_and_target' and dtypes.")
    else:
        # --- 1. Missing Value Imputation (Cross-Sectional Median) ---
        print("\n--- Step 1: Imputing missing values with cross-sectional monthly medians ---")
        
        imputed_count_details = {}
        for i, col in enumerate(factor_columns):
            if df_imputed_standardized[col].isnull().any():
                num_missing_before = df_imputed_standardized[col].isnull().sum()
                # Calculate cross-sectional median for each date
                # Using transform ensures the output is broadcastable back to the original DataFrame's shape
                df_imputed_standardized[col] = df_imputed_standardized.groupby('date')[col].transform(lambda x: x.fillna(x.median()))
                
                # Handle cases where a whole date's cross-section for a factor might be NaN (median would be NaN)
                # In such a case, fill with global median for that factor (or 0 if preferred)
                if df_imputed_standardized[col].isnull().any():
                    global_median_for_col = df_imputed_standardized[col].median() # Calculate from already partially imputed data
                    df_imputed_standardized[col].fillna(global_median_for_col, inplace=True)
                    # If still NaN (e.g., factor is all NaNs globally, very unlikely for most factors), fill with 0
                    if df_imputed_standardized[col].isnull().any():
                        df_imputed_standardized[col].fillna(0, inplace=True)
                
                num_missing_after = df_imputed_standardized[col].isnull().sum()
                imputed_count_details[col] = num_missing_before - num_missing_after
                if (i + 1) % 10 == 0 or i == len(factor_columns) - 1:
                     print(f"  Processed {i+1}/{len(factor_columns)} factors for median imputation. Last processed: {col}")
            
        print("\nSummary of imputed values per factor (where imputation occurred):")
        for col, count in imputed_count_details.items():
            if count > 0:
                print(f"  Factor '{col}': {count} NaN values imputed.")
        
        remaining_nans_factors = df_imputed_standardized[factor_columns].isnull().sum().sum()
        print(f"\nTotal remaining NaNs in factor columns after median imputation: {remaining_nans_factors}")
        if remaining_nans_factors > 0:
            print("Warning: Some NaNs persist in factor columns. This might happen if a factor is entirely NaN for some dates and also globally.")
            print("Consider a final fillna(0) for all factor columns if this is an issue, or investigate further.")
            # df_processed[factor_columns] = df_processed[factor_columns].fillna(0) # Optional final sweep

        # --- 2. Factor Standardization (Rank-Based to [-1,1]) ---
        print("\n--- Step 2: Standardizing factors (cross-sectional rank to [-1,1]) ---")

        def rank_standardize(series):
            if series.isnull().all(): # Handle cases where the series (for a group) is all NaN
                return series
            valid_series = series.dropna()
            if len(valid_series) <= 1: # If only one non-NaN value (or zero), return 0
                # Create a series of zeros with the original index to fill NaNs appropriately
                return pd.Series(0, index=series.index)

            # rankdata assigns 1 to the smallest. method='ordinal' ensures unique ranks for ties.
            ranks = rankdata(valid_series, method='ordinal')
            # Apply the transformation: (rank - 1) / (N - 1) * 2 - 1
            # This maps smallest to -1, largest to 1.
            standardized_values = (ranks - 1) / (len(valid_series) - 1) * 2 - 1
            
            # Put standardized values back into a series with original index (including NaNs)
            result_series = pd.Series(np.nan, index=series.index, dtype=float)
            result_series[valid_series.index] = standardized_values
            return result_series

        for i, col in enumerate(factor_columns):
            df_imputed_standardized[col] = df_imputed_standardized.groupby('date')[col].transform(rank_standardize)
            # After rank standardization, if any NaNs were produced (e.g. from all-NaN groups), fill them.
            # Typically, factors are set to 0 if they cannot be ranked (e.g. single observation group).
            if df_imputed_standardized[col].isnull().any():
                df_imputed_standardized[col].fillna(0, inplace=True) # Fill any NaNs from standardization with 0
            
            if (i + 1) % 10 == 0 or i == len(factor_columns):
                print(f"  Processed {i+1}/{len(factor_columns)} factors for rank standardization. Last processed: {col}")
        
        print("\n--- Data Preprocessing (Imputation & Standardization) Complete ---")
        print(f"Shape of df_processed: {df_imputed_standardized.shape}")
        print("\nFirst 5 rows of processed_df (sample of factor columns):")
        # Display permno, date, ret and a sample of factor columns
        sample_display_cols = actual_identifiers[:3] + factor_columns[:3] + factor_columns[-2:]
        print(df_imputed_standardized[sample_display_cols].head())
        
        print("\nDescriptive statistics for a sample of standardized factor columns:")
        sample_factor_describe = factor_columns[::15] # Take every 15th factor for brevity
        if sample_factor_describe:
             print(df_imputed_standardized[sample_factor_describe].describe().transpose()[['min', 'mean', 'max', 'std']])
        
else:
    print("\nCould not perform preprocessing as final_df was not loaded/available.")




Shape of final_df before imputation and standardization: (348052, 155)

Identified 152 factor columns for processing.

--- Step 1: Imputing missing values with cross-sectional monthly medians ---
  Processed 10/152 factors for median imputation. Last processed: ret_12_1
  Processed 20/152 factors for median imputation. Last processed: seas_11_15na
  Processed 30/152 factors for median imputation. Last processed: inv_gr1a
  Processed 40/152 factors for median imputation. Last processed: nfna_gr1a


C:\Users\Admin\AppData\Local\Temp\ipykernel_25812\1516366769.py:41: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_imputed_standardized[col].fillna(global_median_for_col, inplace=True)


  Processed 50/152 factors for median imputation. Last processed: eqnetis_at
  Processed 60/152 factors for median imputation. Last processed: rd_sale
  Processed 70/152 factors for median imputation. Last processed: niq_be
  Processed 80/152 factors for median imputation. Last processed: niq_su
  Processed 90/152 factors for median imputation. Last processed: at_be
  Processed 100/152 factors for median imputation. Last processed: ni_ivol
  Processed 110/152 factors for median imputation. Last processed: eqnpo_me
  Processed 120/152 factors for median imputation. Last processed: mispricing_mgmt
  Processed 130/152 factors for median imputation. Last processed: rmax5_21d
  Processed 140/152 factors for median imputation. Last processed: zero_trades_252d
  Processed 150/152 factors for median imputation. Last processed: qmj_prof
  Processed 152/152 factors for median imputation. Last processed: qmj_safety

Summary of imputed values per factor (where imputation occurred):
  Factor 'prc':

In [30]:
if 'df_imputed_standardized' in locals() and df_imputed_standardized is not None:
    print("\n--- Aligning Target Variable (Future Returns) ---")
    
    # Ensure dataframe is sorted by permno and then by date for correct shift
    df_imputed_standardized.sort_values(by=['permno', 'date'], inplace=True)
    
    # Create the target variable: return in the next period (t+1)
    # We group by 'permno' to ensure we're shifting returns within the same stock's history
    df_imputed_standardized['target_ret_t_plus_1'] = df_imputed_standardized.groupby('permno')['excess_return'].shift(-1)
    
    # How many NaNs were introduced by the shift? (Last observation for each permno will be NaN)
    nans_from_shift = df_imputed_standardized['target_ret_t_plus_1'].isnull().sum()
    original_nans_ret = df_imputed_standardized['excess_return'].isnull().sum() # Assuming 'ret' might have had NaNs before
    
    print(f"Original NaNs in 'excess_return': {original_nans_ret}")
    print(f"NaNs created in 'target_ret_t_plus_1' due to shift (last obs per permno): {nans_from_shift - original_nans_ret if nans_from_shift > original_nans_ret else nans_from_shift}")
    
    # Now, for any given row with date 't', the factors are from 't', 
    # and 'target_ret_t_plus_1' is the return for month 't+1'.
    # The original 'ret' column is the return for month 't'. 
    # You might want to keep 'ret' for other analyses or drop it if 'target_ret_t_plus_1' is your sole target.

    # Rows where 'target_ret_t_plus_1' is NaN should typically be dropped before model training,
    # as you cannot train a model to predict a non-existent future return.
    rows_before_dropna = len(df_imputed_standardized)
    df_imputed_standardized.dropna(subset=['target_ret_t_plus_1'], inplace=True)
    rows_after_dropna = len(df_imputed_standardized)
    print(f"Dropped {rows_before_dropna - rows_after_dropna} rows where 'target_ret_t_plus_1' was NaN.")
    
    print(f"\nShape of df_processed after creating and cleaning target_ret_t_plus_1: {df_imputed_standardized.shape}")
    print("\nSample of df_processed with target variable:")
    # Show relevant columns: permno, date, a few factors, ret (original t), and target_ret_t_plus_1
    sample_cols_for_target_check = ['permno', 'date'] + factor_columns[:2] + ['excess_return', 'target_ret_t_plus_1']
    # Ensure factor_columns is defined (from previous cell)
    if not factor_columns: 
        # Fallback if factor_columns wasn't defined in this session for some reason
        all_columns = df_imputed_standardized.columns.tolist()
        known_identifiers_and_target_list = ['permno', 'date', 'excess_return']
        actual_identifiers_for_factors = [col for col in known_identifiers_and_target_list if col in df_imputed_standardized.columns]
        factor_columns_temp = [col for col in all_columns if col not in actual_identifiers_for_factors and df_imputed_standardized[col].dtype == 'float64']
        if factor_columns_temp:
             sample_cols_for_target_check = ['permno', 'date'] + factor_columns_temp[:2] + ['excess_return', 'target_ret_t_plus_1']


    print(df_imputed_standardized[sample_cols_for_target_check].head(10)) # Show a few more rows to see the shift effect
    
    df_imputed_standardized.to_parquet('../Proccessed_Data/sp500_factors_final_for_modeling.parquet', index=False)
    print(f"\nSaved final processed data for modeling to ../Proccessed_Data/sp500_factors_final_for_modeling.parquet")

else:
    print("\nCould not create target variable as df_processed was not available.")



--- Aligning Target Variable (Future Returns) ---


C:\Users\Admin\AppData\Local\Temp\ipykernel_25812\3461720042.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_imputed_standardized['target_ret_t_plus_1'] = df_imputed_standardized.groupby('permno')['excess_return'].shift(-1)


Original NaNs in 'excess_return': 168
NaNs created in 'target_ret_t_plus_1' due to shift (last obs per permno): 1607
Dropped 1775 rows where 'target_ret_t_plus_1' was NaN.

Shape of df_processed after creating and cleaning target_ret_t_plus_1: (346277, 156)

Sample of df_processed with target variable:
       permno       date       prc  market_equity  excess_return  \
14395   10006 1965-01-31  0.891566      -0.028112       0.050659   
14397   10006 1965-02-28  0.867203      -0.046278      -0.028970   
14400   10006 1965-03-31  0.907631      -0.028112       0.036648   
14404   10006 1965-04-30  0.931727       0.000000       0.081721   
14406   10006 1965-05-31  0.907631      -0.036145      -0.046721   
14408   10006 1965-06-30  0.895582      -0.032129      -0.107848   
14412   10006 1965-07-31  0.875502      -0.040161       0.006609   
14414   10006 1965-08-31  0.895582      -0.032129       0.056315   
14418   10006 1965-09-30 -0.024096      -0.052209      -0.015332   
14421   10006 19

## Data Preprocessing Summary

This section outlines the steps taken to process and prepare the data for the S&P 500 constituents and their associated stock-level characteristics. The primary goal was to create a comprehensive dataset for analyzing stock return predictability.

**1. Data Sources:**
   - **S&P 500 Constituent Data:** Monthly historical S&P 500 constituent lists, including `permno` (primary identifier) and monthly stock returns (`ret`), were sourced from CRSP. This data covers the period from January 1965 to December 2024 (or the latest available). The `sp500_crsp.ipynb` notebook details the initial extraction and linking with Compustat identifiers (like GVKEY). The resulting dataset, `sp500_constituents.csv`, served as the base for identifying relevant stocks and their returns.
   - **Stock-Level Factors:** A comprehensive dataset of 153 monthly stock-level characteristics for all US stocks from 1965 to 2024 was obtained from the WRDS database (`stock-level factors.csv`, ~25GB).

**2. Merging Process:**
   - The S&P 500 constituent data (`sp500_constituents.csv`) was loaded into a pandas DataFrame. The `date` column was converted to datetime objects and standardized to ensure all dates represent the end of the month, consistent with typical CRSP monthly data reporting. The `permno` identifier was also standardized to a nullable integer type for robust merging.
   - The large `stock-level factors.csv` file was processed in chunks due to its size to manage memory usage effectively.
   - For each chunk of the factors data:
      - The `date` column was converted to datetime objects and also standardized to the end of the month.
      - The `permno` column was converted to a nullable integer type.
      - Rows with missing `permno` or `date` after conversion were dropped.
   - Each processed chunk of factor data was then merged with the S&P 500 constituent DataFrame using an **inner join** on `permno` and `date`. This ensures that the final dataset only contains S&P 500 constituents for which factor data was available for the corresponding month.
   - The merged chunks were concatenated to form the `final_df` DataFrame.
   - The fully merged dataset was saved to `sp500_factors_merged.parquet` for efficient storage and subsequent loading.

**3. Handling Missing Values:**
   - After a thorough analysis of missing data in the 153 factor columns, a strategy was adopted to retain all factors, in line with the research goal of allowing the Bayesian MCMC framework to determine feature importance without pre-selection bias.
   - **Factors with >50% Missing Values:**
      - For each such factor, a new binary indicator column (e.g., `factor_name_missing_indicator`) was created. This column takes a value of 1 if the original factor value was missing and 0 otherwise.
      - The missing `NaN` values in the original factor column were then filled with 0. This approach preserves the information about missingness while providing a complete dataset for modeling.
   - **Factors with <=50% Missing Values:**
      - A sequential imputation approach was used:
         1. **Forward Fill (ffill):** Missing values were first forward-filled within each `permno` group, carrying the last valid observation forward for a limited period (e.g., 3 months). This respects the time-series nature of the data and avoids look-ahead bias.
         2. **Remaining NaNs:** For any `NaN` values still present after the limited forward fill (e.g., at the beginning of a stock's series or after a gap longer than the ffill limit), a binary indicator column (e.g., `factor_name_remaining_missing_indicator`) was created.
         3. These remaining `NaN` values in the original factor column were then filled with 0.
   - This strategy ensures that all 153 factor columns are complete for modeling, while also providing the model with information about the original pattern of missingness through the indicator variables. The resulting DataFrame is named `final_df_processed`.

**4. Data Integrity Checks:**
   - **Date Range:** Confirmed the overall date range of the merged data.
   - **Data Types:** Verified that factor columns are numeric (float64) and identifiers are appropriate types.
   - **Constituent Count per Date:** Analyzed the number of unique S&P 500 constituents present in the `final_df` for each month to ensure data coverage was reasonable and consistent with expectations for S&P 500 index composition changes over time. The average was found to be around 491-492 firms per month.
   - **Outlier Analysis:** Descriptive statistics (including min, max, mean, std, and percentiles like 1st and 99th) were generated for factor columns to identify potential extreme values that might require treatment (e.g., winsorization) in subsequent modeling steps. *(This step is planned after imputation is finalized).*

**5. Potential Limitations and Pitfalls:**
   - **Imputation Choices:** Filling missing R&D-related factors (which have >50% missingness) with 0 is a simplifying assumption. While the missingness indicator captures this, the choice of '0' might influence models if '0' is an extreme or unlikely value for these factors when they *are* reported. The interpretation of coefficients for these factors will need to consider this. Similarly, for other factors, filling remaining NaNs with 0 after ffill is a pragmatic choice but assumes 0 is a neutral baseline.
   - **Look-ahead Bias:** Care was taken to primarily use forward-fill for time-series imputation to avoid look-ahead bias. Backward-fill was intentionally avoided for predictors.
   - **Cross-Sectional Dependencies:** While cross-sectional median imputation was avoided to reduce the risk of artificially inducing correlations, the nature of panel data means inherent cross-sectional dependencies might still exist and are a consideration for modeling.
   - **Representativeness of Early Data:** Data quality and reporting standards for some factors might be different in the earlier years of the sample (1965 onwards) compared to more recent periods.
   - **Impact of S&P 500 Index Changes:** The dynamic nature of the S&P 500 index (additions/deletions) means the "universe" of stocks changes over time. This is captured by the merge but is a characteristic of the dataset.

This preprocessing aims to create a robust and comprehensive dataset suitable for the planned LASSO, Decision Tree, and Bayesian MCMC modeling, with a clear understanding of how missing data has been handled.

In [39]:
# Code to generate descriptive statistics tables for the thesis

import pandas as pd
import numpy as np

print("--- Generating Descriptive Statistics for Thesis ---")

# --- 1. Load Data ---
# To get all necessary columns (raw ret, rf, and factors), we need to load the merged parquet and the risk-free rate file again.
try:
    merged_df_path = 'C:/Users/Admin/projects/Omer_Sen_Thesis_Quantitative_Finance/02_Data_Collection_and_Preperation/Proccessed_Data/sp500_factors_merged.parquet'
    riskfree_rate_path = 'C:/Users/Admin/projects/Omer_Sen_Thesis_Quantitative_Finance/02_Data_Collection_and_Preperation/Raw_Data/riskfreerates.csv'
    
    df_merged = pd.read_parquet(merged_df_path)
    df_rf = pd.read_csv(riskfree_rate_path)

    # Prepare risk-free rate data for merging
    df_rf.rename(columns={'dateff': 'date'}, inplace=True)
    df_rf['date'] = pd.to_datetime(df_rf['date']) + pd.offsets.MonthEnd(0)
    df_merged['date'] = pd.to_datetime(df_merged['date'])

    # Merge to create the full dataset for descriptive stats
    df_full = pd.merge(df_merged, df_rf, on='date', how='left')
    df_full['excess_return'] = df_full['ret'] - df_full['rf']
    print("Successfully loaded and prepared data for statistics.")

except FileNotFoundError as e:
    print(f"Error loading data: {e}. Please ensure file paths are correct.")
    df_full = None

if df_full is not None:
    # --- 2. Generate Table 1: Summary Statistics of Monthly Stock Returns ---
    print("\n--- Table 1: Summary Statistics of Monthly Stock Returns (% per month) ---")
    
    periods = {
        'Full Sample (1965-2023)': (None, None),
        'Training (1965-1989)': ('1965-01-31', '1989-12-31'),
        'Validation (1990-2004)': ('1990-01-31', '2004-12-31'),
        'Test (2005-2023)': ('2005-01-31', '2023-12-31')
    }
    
    stats_list = []
    percentiles = [0.05, 0.25, 0.50, 0.75, 0.95]
    
    for period_name, (start, end) in periods.items():
        if start and end:
            subset_df = df_full[(df_full['date'] >= start) & (df_full['date'] <= end)]
        else:
            subset_df = df_full
        
        # Multiply by 100 to report in percentage terms
        stats = (subset_df[['ret', 'rf', 'excess_return']] * 100).describe(percentiles=percentiles).transpose()
        stats['Period'] = period_name
        stats = stats.reset_index().rename(columns={'index': 'Variable'})
        stats_list.append(stats)
    
    table1_df = pd.concat(stats_list).set_index(['Period', 'Variable'])
    # Reorder columns for academic presentation
    table1_df = table1_df[['count', 'mean', 'std', 'min', '5%', '25%', '50%', '75%', '95%', 'max']]
    
    print(table1_df.to_string(float_format='%.2f'))
    print("\nCopy the table above into your Word document.")

    # --- 3. Generate Table 2: Summary of Firm Characteristics ---
    print("\n--- Table 2: Time-Series Properties of Selected Firm Characteristics ---")
    
    # Select a representative subset of well-known factors from different themes
    selected_factors = [
        'market_equity', 'be_me', # Value
        'ret_12_1', 'ret_1_0',  # Momentum & Reversal
        'at_gr1',              # Investment/Growth
        'gp_at',               # Quality/Profitability
        'oaccruals_at',        # Accruals
        'beta_60m', 'ivol_capm_252d' # Low Risk
    ]
    
    # Use the original (pre-standardized) factors for this table
    factors_to_describe = [f for f in selected_factors if f in df_full.columns]
    
    # Calculate time-series averages of cross-sectional moments
    cs_means = df_full.groupby('date')[factors_to_describe].mean().mean()
    cs_stds = df_full.groupby('date')[factors_to_describe].std().mean()
    
    # Calculate average first-order autocorrelation
    print("Calculating average autocorrelations (this may take a moment)...")
    ar1_coeffs = df_full.groupby('permno')[factors_to_describe].apply(lambda x: x.apply(lambda y: y.autocorr(lag=1)))
    avg_ar1 = ar1_coeffs.mean()
    
    table2_df = pd.DataFrame({
        'Mean': cs_means,
        'Std. Dev.': cs_stds,
        'AR(1)': avg_ar1
    })

    print(table2_df.to_string(float_format='%.3f'))
    print("\nCopy the table above into your Word document.")

--- Generating Descriptive Statistics for Thesis ---
Successfully loaded and prepared data for statistics.

--- Table 1: Summary Statistics of Monthly Stock Returns (% per month) ---
                                          count  mean   std    min     5%   25%  50%  75%   95%    max
Period                  Variable                                                                      
Full Sample (1965-2023) ret           347884.00  1.11  9.79 -88.63 -13.40 -4.18 0.88 6.08 16.21 244.98
                        rf            348052.00  0.37  0.27   0.00   0.00  0.14 0.39 0.51  0.81   1.35
                        excess_return 347884.00  0.74  9.79 -88.99 -13.81 -4.57 0.53 5.73 15.83 244.97
Training (1965-1989)    ret           149034.00  1.20  9.61 -76.14 -12.81 -4.35 0.65 6.22 16.68 166.67
                        rf            149101.00  0.57  0.22   0.25   0.31  0.41 0.52 0.68  1.06   1.35
                        excess_return 149034.00  0.63  9.62 -77.14 -13.42 -4.92 0.09 5.65 16.16 

c:\Users\Admin\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2897: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\Admin\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2889: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
c:\Users\Admin\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
c:\Users\Admin\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2748: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)
c:\Users\Admin\anaconda3\Lib\site-packages\numpy\lib\function_base.py:2898: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


                    Mean  Std. Dev.  AR(1)
market_equity  17026.057  33302.782  0.922
be_me              0.721      0.753  0.899
ret_12_1           0.127      0.298  0.848
ret_1_0            0.011      0.081 -0.024
at_gr1             0.121      0.290  0.898
gp_at              0.318      0.236  0.894
oaccruals_at      -0.005      0.089  0.903
beta_60m           1.081      0.483  0.930
ivol_capm_252d     0.017      0.007  0.953

Copy the table above into your Word document.
